# Playground: download and unpack random papers

Draws random papers from the metadata, downloads their v1 source with
`language_shift.arxiv_source` and shows what came out.
Files land in `data/sources/raw/` and `data/sources/unpacked/`
(gitignored). Each download waits 3 s (arXiv's rule).

In [26]:
import random
from pathlib import Path

import pandas as pd

from language_shift.arxiv_source import (
    NotLatex,
    SourceNotAvailable,
    prepare_paper,
)

ROOT = Path.cwd().parent if Path.cwd().name == "dev" else Path.cwd()
META = ROOT / "data" / "metadata" / "arxiv_metadata.csv"

## Metadata

In [27]:
meta = pd.read_csv(
    META,
    usecols=["id", "title", "primary_category", "group", "v1_date", "n_versions", "v1_size_kb"],
    dtype=str,
    keep_default_na=False,
)
candidates = meta[
    meta["group"].isin(["cs", "math", "physics"])
    & (meta["v1_date"] >= "2005-09-01")
    & (meta["v1_date"] < "2026-11-01")
]
len(candidates)

2671551

## Draw papers
Change `N`, or set `SEED` to a number to repeat a draw.

In [28]:
N = 20
SEED = None  # None = new random draw each run

seed = SEED if SEED is not None else random.randrange(10**6)
sample = candidates.sample(N, random_state=seed)
print("seed:", seed)
sample

seed: 965958


,id,title,primary_category,group,v1_date,n_versions,v1_size_kb
561443,0909.4999,Gamma-ray astronomy in the summer of 2009,astro-ph.HE,physics,2009-09-28,1,70
1327089,1711.05930,General rogue waves in the nonlocal PT-symmetr...,nlin.SI,physics,2017-11-16,1,2953
513162,0812.2954,Negative Effective Density in An Acoustic Meta...,cond-mat.mtrl-sci,physics,2008-12-16,3,61
1798114,2011.11375,The Diary of Niels: Affective engagement throu...,cs.HC,cs,2020-11-23,1,9215
2842184,2509.26087,Easy3D-Labels: Supervising Semantic Occupancy ...,cs.CV,cs,2025-09-30,5,7744
2137004,2210.04617,Topological Effects on Non-Relativistic Eigenv...,hep-th,physics,2022-10-04,2,192
2283350,2306.16728,IoT Data Processing for Smart City and Semanti...,cs.SE,cs,2023-06-29,1,6376
1984801,2112.01212,Curvature of the Second kind and a conjecture ...,math.DG,math,2021-12-02,1,14
1935851,2108.13139,CSDS: A Fine-Grained Chinese Dataset for Custo...,cs.CL,cs,2021-08-30,2,393
2311201,2308.09816,Comparisons and Predictions for Collisions of ...,nucl-th,physics,2023-08-18,4,4058


## Download and unpack

In [29]:
papers, excluded, failed = [], [], []
for arxiv_id in sample["id"]:
    try:
        papers.append(prepare_paper(arxiv_id))
    except NotLatex as e:
        excluded.append(str(e))
    except SourceNotAvailable as e:
        failed.append(str(e))

summary = pd.DataFrame([
    {
        "id": p.id,
        "kind": p.kind,
        "files": len(p.files),
        "tex_files": len(p.tex_files),
        "main_tex": str(p.main_tex) if p.main_tex else None,
        "raw": p.raw_path.name,
    }
    for p in papers
])
print("excluded:", excluded)
print("failed:", failed)
summary

excluded: ['2306.16728v1: PDF only, excluded']
failed: []


,id,kind,files,tex_files,main_tex,raw
0,0909.4999,tar,2,1,torres-rapporteur-talk-icrc09.tex,0909.4999v1.gz
1,1711.05930,tar,36,1,arxiv.tex,1711.05930v1.gz
2,0812.2954,tar,4,1,meta_density_1.tex,0812.2954v1.gz
3,2011.11375,tar,14,2,main.tex,2011.11375v1.gz
4,2509.26087,tar,21,4,ojits.tex,2509.26087v1.gz
5,2210.04617,tar,27,1,template.tex,2210.04617v1.gz
6,2112.01212,tar,2,1,main.tex,2112.01212v1.gz
7,2108.13139,tar,10,1,emnlp2021.tex,2108.13139v1.gz
8,2308.09816,tar,32,1,UU_paper.tex,2308.09816v1.gz
9,2606.14127,tar,6,1,core_arxiv.tex,2606.14127v1.gz


## Look inside
File list and the start of the main `.tex` file per paper.

In [30]:
LINES = 40

for p in papers:
    print("=" * 80)
    print(p.id, "|", p.kind, "|", p.dir)
    print("files:", [str(f) for f in p.files][:30])
    if p.main_path:
        text = p.main_path.read_text(encoding="utf-8", errors="replace")
        print(f"--- {p.main_tex} ({len(text):,} chars) ---")
        print("\n".join(text.splitlines()[:LINES]))

0909.4999 | tar | /Users/je61usyl/projects/Language_Shift_In_Papers/data/sources/unpacked/0909.4999v1
files: ['icrc2009.cls', 'torres-rapporteur-talk-icrc09.tex']
--- torres-rapporteur-talk-icrc09.tex (56,257 chars) ---
%ICRC2009 proceedings template 

\documentclass{icrc2009}

\usepackage{graphicx}   % for including figures
\usepackage{caption}    % for captions
\usepackage[font=footnotesize]{subfig} % subfig.sty for a double column floating figure using two subfigures
\usepackage{fixltx2e}
%\usepackage{stfloats}
%\fnbelowfloat
\usepackage{url}

\newcommand{\shorttitle}[1]%
{\markboth{Proceedings of the 31\MakeLowercase{$^{st}$} ICRC, {\L}\'{o}d\'{z} 2009}{#1} }
\newcommand{\etal}{\MakeLowercase{\textit{et al. }}} % "et al."




\begin{document}
\title{$\gamma$-ray astronomy in the summer of 2009}

\author{\IEEEauthorblockN{Diego F. Torres\IEEEauthorrefmark{1}
			}
                            \\
\IEEEauthorblockA{\IEEEauthorrefmark{1}ICREA \& Institut de Ci\`encies de l'Espai (IEEC-CS

## Notes